In [ ]:
## PRICE ELASTICITIES BY ITEM FROM LightGBM

- LightGBM cannot give a coefficient to Price as Linear Regression (like OLS) gives:
- LightGBM learns demand as a nonlinear function: Q=f(P,lags,calendar,store,item,… )
- So there is no single parameter for price.

- But local elasticities can be calculated from the model by simulation.

# Local Elasticites
The standard approach is price perturbation simulation.

For each observation:

-1 predict baseline demand
-2 increase price slightly
-3 recompute predicted demand
-4 calculate elasticity

Elasticity formula:{Delta Q/Q}{\Delta P/P}

In [10]:
import joblib
import pandas as pd
import numpy as np

# 1) load model
model = joblib.load("models/lightgbm_model_base.joblib")

print(type(model))
df_train=pd.read_csv("data/df_train.csv")
df_test=pd.read_csv("data/df_test.csv")
simulator_df = pd.concat([df_train, df_test], axis=0, ignore_index=True)
simulator_df.to_csv("data/simulator_df.csv", index=False)

<class 'lightgbm.sklearn.LGBMRegressor'>


In [11]:
simulator_df = pd.read_csv("data/simulator_df.csv")

cat_cols = [
    "item_id", "dept_id", "cat_id", "store_id", "state_id",
    "weekday", "event_name_1", "event_type_1", "event_name_2", "event_type_2"
]

for c in cat_cols:
    simulator_df[c] = simulator_df[c].astype("category")

/tmp/ipykernel_104437/2451262483.py:1: DtypeWarning: Columns (13) have mixed types. Specify dtype option on import or set low_memory=False.
  simulator_df = pd.read_csv("data/simulator_df.csv")


In [12]:
simulator_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 279208 entries, 0 to 279207
Data columns (total 38 columns):
 #   Column              Non-Null Count   Dtype   
---  ------              --------------   -----   
 0   item_id             279208 non-null  category
 1   dept_id             279208 non-null  category
 2   cat_id              279208 non-null  category
 3   store_id            279208 non-null  category
 4   state_id            279208 non-null  category
 5   wm_yr_wk            279208 non-null  int64   
 6   weekday             279208 non-null  category
 7   wday                279208 non-null  int64   
 8   month               279208 non-null  int64   
 9   year                279208 non-null  int64   
 10  event_name_1        279208 non-null  category
 11  event_type_1        22616 non-null   category
 12  event_name_2        279208 non-null  category
 13  event_type_2        581 non-null     category
 14  snap_CA             279208 non-null  int64   
 15  snap_TX          

In [ ]:
import numpy as np
import pandas as pd

def simulate_item_prices(
    model,
    X,
    item_id,
    price_grid,
    price_col="sell_price",
    item_col="item_id",
    inventory=None,
    clip_negative_demand=True
):
    subset = X[X[item_col] == item_id].copy()

    if subset.empty:
        return pd.DataFrame()

    results = []

    for p in price_grid:
        X_sim = subset.copy()
        X_sim[price_col] = p

        # update price-derived features if present
        if "price_change_1" in X_sim.columns:
            if "price_lag_1" in X_sim.columns:
                X_sim["price_change_1"] = X_sim[price_col] - X_sim["price_lag_1"]
            else:
                X_sim["price_change_1"] = 0.0

        if "price_pct_change_1" in X_sim.columns:
            if "price_lag_1" in X_sim.columns:
                denom = X_sim["price_lag_1"].replace(0, np.nan)
                X_sim["price_pct_change_1"] = (X_sim[price_col] - X_sim["price_lag_1"]) / denom
                X_sim["price_pct_change_1"] = X_sim["price_pct_change_1"].fillna(0.0)
            else:
                X_sim["price_pct_change_1"] = 0.0

        if "price_rel" in X_sim.columns and "price_roll_mean_28" in X_sim.columns:
            denom = X_sim["price_roll_mean_28"].replace(0, np.nan)
            X_sim["price_rel"] = X_sim[price_col] / denom
            X_sim["price_rel"] = X_sim["price_rel"].fillna(1.0)

        pred_qty = np.asarray(model.predict(X_sim), dtype=float)

        if clip_negative_demand:
            pred_qty = np.clip(pred_qty, 0, None)

        total_qty = pred_qty.sum()

        feasible_qty = min(total_qty, inventory) if inventory is not None else total_qty
        revenue = p * feasible_qty

        results.append({
            "item_id": item_id,
            "candidate_price": p,
            "predicted_qty": total_qty,
            "feasible_qty": feasible_qty,
            "predicted_revenue": revenue
        })

    return pd.DataFrame(results)


def optimize_prices_all_items(
    model,
    X,
    price_col="sell_price",
    item_col="item_id",
    inventory_df=None,
    inventory_item_col="item_id",
    inventory_qty_col="inventory",
    grid_low=0.8,
    grid_high=1.2,
    n_prices=9,
    clip_negative_demand=True
):
    """
    Simulate candidate prices for all items and return optimal prices.

    Parameters
    ----------
    model : fitted model
    X : pd.DataFrame
        Dataset used for simulation, e.g. X_test or latest item-store-week slice.
    price_col : str
        Price column name.
    item_col : str
        Item identifier column.
    inventory_df : pd.DataFrame or None
        Optional dataframe with inventory by item.
    inventory_item_col : str
        Item column in inventory_df.
    inventory_qty_col : str
        Inventory quantity column in inventory_df.
    grid_low, grid_high : float
        Lower and upper multipliers around current price.
    n_prices : int
        Number of candidate prices.
    clip_negative_demand : bool
        Whether to clip negative predicted demand to zero.

    Returns
    -------
    all_simulations : pd.DataFrame
    best_prices : pd.DataFrame
    """
    all_results = []

    item_ids = X[item_col].dropna().unique()

    inventory_map = None
    if inventory_df is not None:
        inventory_map = dict(zip(inventory_df[inventory_item_col], inventory_df[inventory_qty_col]))

    for item_id in item_ids:
        subset = X[X[item_col] == item_id]

        if subset.empty:
            continue

        current_price = subset[price_col].median()

        if pd.isna(current_price) or current_price <= 0:
            continue

        price_grid = np.round(
            np.linspace(current_price * grid_low, current_price * grid_high, n_prices),
            2
        )

        inventory = None
        if inventory_map is not None:
            inventory = inventory_map.get(item_id, None)

        sim_df = simulate_item_prices(
            model=model,
            X=X,
            item_id=item_id,
            price_grid=price_grid,
            price_col=price_col,
            item_col=item_col,
            inventory=inventory,
            clip_negative_demand=clip_negative_demand
        )

        if not sim_df.empty:
            sim_df["current_price"] = current_price
            all_results.append(sim_df)

    if not all_results:
        return pd.DataFrame(), pd.DataFrame()

    all_simulations = pd.concat(all_results, ignore_index=True)

    best_idx = all_simulations.groupby("item_id")["predicted_revenue"].idxmax()
    best_prices = (
        all_simulations.loc[best_idx]
        .sort_values("predicted_revenue", ascending=False)
        .reset_index(drop=True)
    )

    best_prices["price_change_pct"] = (
        (best_prices["candidate_price"] - best_prices["current_price"]) / best_prices["current_price"]
    )

    return all_simulations, best_prices

In [ ]:
all_simulations, best_prices = optimize_prices_all_items(
    model=baseline_model,
    X=X_test,
    price_col="sell_price",
    item_col="item_id",
    grid_low=0.8,
    grid_high=1.2,
    n_prices=9
)

print(best_prices.head(10))